# Chapter 1 — Introducing Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saanikakulkarni07/learn-neural-nets/blob/main/ch01-introducing-neural-networks/exercises.ipynb)

Chapter 1 of NNFS is conceptual — it doesn't write a full network yet. So these exercises focus on **building rock-solid intuition** for the one equation everything else is built on:

$$\text{output} = \Big(\sum_i x_i\,w_i\Big) + b$$

By the end you'll be able to *feel* what a weight does, what a bias does, why we need an activation function, and how a single neuron generalizes to a layer. We close with a tiny astrophysics-flavored example.

**Goals**
1. Reason about AI ⊃ ML ⊃ NN ⊃ deep learning.
2. Compute a single neuron's output by hand and in NumPy.
3. See weights as *slope* and biases as *offset*.
4. Understand why a purely linear neuron is limited.


## 0. Setup
Run this first. In Colab it installs the book's `nnfs` helper; locally it's already in `requirements.txt`.

In [1]:
try:
    import nnfs
except ImportError:
    !pip install nnfs -q
    import nnfs
import numpy as np
import matplotlib.pyplot as plt
nnfs.init()  # fixes the random seed + dtype so results are reproducible
print('setup ok — numpy', np.__version__)

setup ok — numpy 2.0.2


## 1. Concept check (no code)

Answer these in the markdown cell below — they're the conceptual core of Chapter 1.

1. Put these in order from most general to most specific: *deep learning, machine learning, neural networks, artificial intelligence*.
2. What makes a neural network *deep*?
3. In the server-failure example, what is a **feature**, a **sample**, and a **label**?
4. **Supervised vs. unsupervised** learning — one sentence each.
5. What's the difference between **classification** and **regression**? Give one astro example of each.
6. What is **overfitting**, and why do we hold out **out-of-sample** data?


_Your answers:_

1. artificial intelligence, machine learning, deep learning, neural networks
2. it goes through many many layers to make a decision
3. feature: a characteristic of the sample, sample: a row of data, label: categorical characterization of data
4. supervised uses labels for the data while unsupervised doesn't, tries to create its own labels
5. classification: assigning data to labels (galaxies), regression: fitting a model function to data (redshifts)
6. overfitting: when the neural net/model starts memorizing data rather than finding patterns in the data. we hold out of sample data to prevent overfitting.


<details><summary>Check yourself (click to reveal)</summary>

1. AI ⊃ ML ⊃ NN ⊃ deep learning.
2. Having **two or more hidden layers**.
3. *Feature* = one measured quantity (e.g. temperature); *sample* = one row / full feature set for one observation; *label* = the known target ("normal" / "failure").
4. Supervised = learn from labeled examples; unsupervised = find structure with no labels.
5. Classification = predict a discrete class (star vs. galaxy vs. QSO); regression = predict a continuous value (photometric redshift).
6. Overfitting = memorizing training data instead of learning the pattern; we hold out data to measure whether the model **generalizes**.
</details>

## 2. A single neuron, by hand

A neuron takes inputs, multiplies each by a **weight**, sums them, and adds a **bias**:
$$\text{output} = (x_1 w_1 + x_2 w_2 + x_3 w_3) + b$$

**Exercise 2a.** Fill in `neuron_output` using *plain Python* (a loop or `zip`, no NumPy yet). Then check it against the assertion.

In [ ]:
inputs  = [1.0, 2.0, 3.0, 2.5]
weights = [0.2, 0.8, -0.5, 1.0]
bias    = 2.0

def neuron_output(inputs, weights, bias):
    # TODO: return sum(x_i * w_i) + bias using plain Python
    total = 0.0
    for x, w in zip(inputs, weights):
        total += x * w
    return total + bias

out = neuron_output(inputs, weights, bias)
print(out)
assert abs(out - 4.8) < 1e-9, 'expected 4.8'
print('✓ matches the book')

**Exercise 2b.** Do the same in **NumPy** with a single dot product. Confirm it equals your hand version. (This is the leap that makes networks fast.)

In [ ]:
inputs  = np.array([1.0, 2.0, 3.0, 2.5])
weights = np.array([0.2, 0.8, -0.5, 1.0])
bias    = 2.0

out_np = np.dot(weights, inputs) + bias  # TODO if blank
print(out_np)
assert np.isclose(out_np, 4.8)
print('✓ dot product == hand-rolled loop')

## 3. Weights are slope, biases are offset

For a single input, the neuron is just a line: $\text{output} = w\,x + b$ — exactly $y = mx + b$. Let's *see* it. Run the cell, then do the mini-exercise.

In [ ]:
x = np.linspace(-2, 2, 100)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))

# Left: vary the WEIGHT (slope), bias fixed at 0
for w in [-0.7, 0.5, 1.0, 2.0]:
    axL.plot(x, w * x + 0.0, label=f'w={w}')
axL.set_title('Weight controls the slope (b=0)')
axL.axhline(0, color='k', lw=0.5); axL.axvline(0, color='k', lw=0.5)
axL.legend(); axL.set_xlabel('input x'); axL.set_ylabel('output')

# Right: vary the BIAS (offset), weight fixed at 1
for b in [-2.0, -0.7, 0.0, 2.0]:
    axR.plot(x, 1.0 * x + b, label=f'b={b}')
axR.set_title('Bias shifts the line up/down (w=1)')
axR.axhline(0, color='k', lw=0.5); axR.axvline(0, color='k', lw=0.5)
axR.legend(); axR.set_xlabel('input x'); axR.set_ylabel('output')
plt.tight_layout(); plt.show()

**Exercise 3.** Without running anything: a neuron has `w = -0.7`, `b = 1.0`. At what input `x` does its output cross zero? Solve $-0.7x + 1.0 = 0$, then verify in code below.

_Your algebra:_ x = ____

In [ ]:
w, b = -0.7, 1.0
x_cross = -b / w  # TODO if blank
print('crosses zero at x =', round(x_cross, 4))
assert np.isclose(w * x_cross + b, 0.0)
print('✓ output is ~0 there')

## 4. Why we'll need activation functions

Chapter 1 hints that real networks don't just use a step function. Here's the key limitation to *feel* now: **stacking linear neurons gives you… another linear function.** Run this — two linear 'layers' collapse into one line.

In [ ]:
x = np.linspace(-2, 2, 100)

# 'Layer 1': two linear neurons
h1 = 2.0 * x + 1.0
h2 = -1.0 * x + 0.5
# 'Layer 2': linear combination of the hidden outputs
y = 0.5 * h1 + 1.5 * h2 - 0.3

# Claim: y is still a straight line in x. Fit a degree-1 poly to check.
slope, intercept = np.polyfit(x, y, 1)
residual = np.max(np.abs(y - (slope * x + intercept)))
print(f'effective line: y = {slope:.3f} x + {intercept:.3f}')
print(f'max deviation from a straight line: {residual:.2e}')
assert residual < 1e-9, 'should be ~0 — linear in, linear out'
print('✓ no matter how many linear layers, it stays linear — hence ReLU/sigmoid (Ch. 4)')

## 5. 🔭 Astro bonus — a one-neuron 'is it bright?' detector

Imagine a single photometric feature: an object's brightness (here, arbitrary units). A single neuron with a hand-set weight and bias, followed by a **step** activation, can act as a crude threshold classifier: *bright source* (1) vs *background* (0).

**Exercise 5.** Set `w` and `b` so the neuron fires (step output 1) only when `brightness > 5`. (Hint: you want `w*brightness + b > 0` ⇔ `brightness > 5`.)

In [ ]:
def step(z):
    return np.where(z > 0, 1, 0)

brightness = np.array([1.0, 3.0, 5.0, 5.2, 8.0, 12.0])

w = 1.0      # TODO
b = -5.0     # TODO: choose so the threshold sits at brightness == 5

z = w * brightness + b
fired = step(z)
print('brightness:', brightness)
print('fired:     ', fired)
expected = (brightness > 5).astype(int)
assert np.array_equal(fired, expected), 'tune w, b so it fires only above 5'
print('✓ your neuron is a brightness threshold detector')

**Reflect:** this is exactly how a trained network *starts* — except instead of you hand-picking `w` and `b`, an optimizer (Ch. 6, 10) will find them from labeled data, and across many neurons it'll learn thresholds far more subtle than 'brightness > 5'. That's the whole journey of this book.

### Stretch (open-ended)
- Add a second feature (e.g. a color index) and make a **two-input** neuron. What does its decision boundary look like in the 2-D feature plane?
- Replace the step with `1/(1+np.exp(-z))` (sigmoid). How does the output change near the threshold? (Foreshadows Ch. 4 & 16.)